# 02 — Lines and Points in PGA2d

This notebook explores the relationship between lines and points in Projective Geometric Algebra. We introduce the **meet** (intersection) and **join** (line through two points) operations, which are fundamental to geometric computations.

## Learning Objectives

- Represent points and lines as multivectors in PGA2d
- Compute line-point incidence using the inner product
- Use the regressive product (meet) for line-line intersection
- Use the outer product (join) for point-point line construction
- Visualize geometric relationships

In [ ]:
# Setup
from amsa import Algebra
import numpy as np
import matplotlib.pyplot as plt

alg = Algebra.pga2d()

## 2.1 Point and Line Representations

In PGA2d:

- **Points** are grade-1 vectors: $P = x e_1 + y e_2 + w e_0$
  - For finite points: $w = 1$ → $(x, y, 1)$
  - For points at infinity: $w = 0$ → direction only

- **Lines** are grade-2 bivectors: $L = a e_1 + b e_2 + c e_0$
  - Represents line $ax + by + c = 0$

The relationship is **duality**: points ↔ lines via the outer product.

In [ ]:
# Create points in PGA2d
p1 = alg.multivector({"e01": 1.0, "e02": 1.0, "e12": 1.0})  # (1, 1)
p2 = alg.multivector({"e01": 3.0, "e02": 1.0, "e12": 1.0})  # (3, 1)
p3 = alg.multivector({"e01": 2.0, "e02": 3.0, "e12": 1.0})  # (2, 3)

print("Point P1 (1, 1):", p1.values)
print("Point P2 (3, 1):", p2.values)
print("Point P3 (2, 3):", p3.values)

# Alternative: using vector constructor (e0 = 1 by default)
p1_vec = alg.vector([1.0, 1.0, 1.0])
print("\nUsing vector constructor:", p1_vec.values)

In [ ]:
# Create lines
# Line: x + y - 2 = 0 → coefficients: a=1, b=1, c=-2
line1 = alg.multivector({"e1": 1.0, "e2": 1.0, "e0": -2.0})

# Line: y - 1 = 0 (horizontal) → a=0, b=1, c=-1
line2 = alg.multivector({"e1": 0.0, "e2": 1.0, "e0": -1.0})

# Line: x = 2 (vertical) → a=1, b=0, c=-2
line3 = alg.multivector({"e1": 1.0, "e2": 0.0, "e0": -2.0})

print("Line L1 (x + y - 2 = 0):", line1.values)
print("Line L2 (y - 1 = 0):", line2.values)
print("Line L3 (x - 2 = 0):", line3.values)

## 2.2 Point-Line Incidence

A point lies on a line when their inner product gives zero:

$$P \cdot L = 0 \Leftrightarrow P \text{ is on } L$$

This is the **incidence test**.

In [ ]:
# Test incidence: is P1 (1, 1) on L1 (x + y - 2 = 0)?
incidence = p1 | line1
print("P1 · L1 =", incidence.values)
print("P1 is on L1:", incidence.component("e") == 0)

# Test another point
p_test = alg.vector([2.0, 0.0, 1.0])  # (2, 0)
incidence2 = p_test | line1
print("\nP(2, 0) · L1 =", incidence2.values)
print("P(2, 0) is on L1:", incidence2.component("e") == 0)

## 2.3 Meet (Intersection of Lines)

The **meet** of two lines is their intersection point. In PGA, we use the **regressive product** for this:

$$P = L_1 \vee L_2$$

This is computed as: $L_1 \vee L_2 = (L_1 \cdot I^{-1}) \cdot (L_2 \cdot I^{-1})$ where $I$ is the pseudoscalar.

In AMSA, we use `.regress(other)` for the meet operation.

In [ ]:
# Meet of two lines
# L1: y = 1 (horizontal through y=1)
# L2: x = 2 (vertical through x=2)

L1 = alg.multivector({"e1": 0.0, "e2": 1.0, "e0": -1.0})
L2 = alg.multivector({"e1": 1.0, "e2": 0.0, "e0": -2.0})

# Meet: intersection point
intersection = L1.regress(L2)

print("Line 1 (y = 1):", L1.values)
print("Line 2 (x = 2):", L2.values)
print("\nIntersection (meet):", intersection.values)
print("\nPoint coordinates: x =", intersection.component("e01"), ", y =", intersection.component("e02"))

In [ ]:
# Visualize the intersection
fig, ax = plt.subplots(figsize=(8, 8))

# Draw L1: y = 1
x_vals = np.linspace(-1, 4, 100)
ax.plot(x_vals, np.ones_like(x_vals), 'b-', linewidth=2, label='L1: y = 1')

# Draw L2: x = 2
ax.plot(np.ones_like(x_vals)*2, x_vals, 'r-', linewidth=2, label='L2: x = 2')

# Draw intersection point
ix = intersection.component("e01") / intersection.component("e12")
iy = intersection.component("e02") / intersection.component("e12")
ax.scatter([ix], [iy], s=200, c='green', zorder=5, marker='*')
ax.text(ix + 0.1, iy + 0.1, f'Intersection ({ix}, {iy})', fontsize=10)

ax.set_xlim(-1, 4)
ax.set_ylim(-1, 4)
ax.set_aspect('equal')
ax.set_title('Meet: Line Intersection', fontsize=12)
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.grid(True, alpha=0.3)
ax.legend()
plt.show()

## 2.4 Join (Line Through Two Points)

The **join** of two points is the line passing through them. We use the **outer product**:

$$L = P_1 \wedge P_2$$

This naturally constructs the line without solving linear equations!

In [ ]:
# Join: line through two points
P1 = alg.vector([1.0, 1.0, 1.0])  # (1, 1)
P2 = alg.vector([3.0, 2.0, 1.0])  # (3, 2)

# Join gives the line
line = P1 ^ P2

print("Point P1 (1, 1):", P1.values)
print("Point P2 (3, 2):", P2.values)
print("\nLine through P1, P2:", line.values)

# Extract line coefficients
a = line.component("e1")
b = line.component("e2")
c = line.component("e0")
print(f"\nLine equation: {a}x + {b}y + {c} = 0")

In [ ]:
# Verify: both points lie on this line
on_line_1 = P1 | line
on_line_2 = P2 | line

print("P1 on line:", on_line_1.component("e") == 0)
print("P2 on line:", on_line_2.component("e") == 0)

## 2.5 Visualizing Join

Let's visualize two points and their join (the line through them).

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))

# Points
ax.scatter([1, 3], [1, 2], s=100, c='blue', zorder=5)
ax.text(1.1, 1.1, 'P1 (1,1)', fontsize=10)
ax.text(3.1, 2.1, 'P2 (3,2)', fontsize=10)

# Draw the line through them
a = line.component("e1")
b = line.component("e2")
c = line.component("e0")

x_vals = np.linspace(-1, 5, 100)
y_vals = -(a * x_vals + c) / b
ax.plot(x_vals, y_vals, 'r-', linewidth=2, label=f'Line: {a}x + {b}y + {c:.1f} = 0')

ax.set_xlim(-1, 5)
ax.set_ylim(-1, 4)
ax.set_aspect('equal')
ax.set_title('Join: Line Through Two Points', fontsize=12)
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.grid(True, alpha=0.3)
ax.legend()
plt.show()

## 2.6 Parallel Lines Meet at Infinity

One of the beautiful properties of PGA: parallel lines meet at a **point at infinity**!

When two lines are parallel, their meet has $e_0 = 0$ — that's the point at infinity in that direction.

In [ ]:
# Two parallel lines
L1 = alg.multivector({"e1": 1.0, "e2": 0.0, "e0": 0.0})   # x = 0
L2 = alg.multivector({"e1": 1.0, "e2": 0.0, "e0": -2.0})  # x = 2

# Meet (intersection)
parallel_meet = L1.regress(L2)

print("Parallel lines: x = 0 and x = 2")
print("Meet:", parallel_meet.values)
print("\ne0 component:", parallel_meet.component("e0"))
print("This is a point at infinity (e0 = 0)!" if parallel_meet.component("e0") == 0 else "This is a finite point")

## 2.7 Summary

We covered:

- **Points**: Grade-1 vectors with $e_0 = 1$ (finite) or $e_0 = 0$ (at infinity)
- **Lines**: Grade-2 bivectors representing $ax + by + c = 0$
- **Incidence test**: $P \cdot L = 0$ checks if point is on line
- **Meet** (`.regress()`): Intersection of two lines → point
- **Join** (`^`): Line through two points → line
- **Parallel lines**: Meet at points at infinity ($e_0 = 0$)

In the next notebook, we'll explore **motors** — the PGA representation of rigid body motions.

## Exercises

### ⭐ Easy

**2.1** Create points P1 = (0, 0) and P2 = (1, 1). Find the line through them using the join (outer product). Verify both points lie on this line.

In [ ]:
# Your turn: ⭐ Exercise 2.1
P1 = alg.vector([0.0, 0.0, 1.0])
P2 = alg.vector([1.0, 1.0, 1.0])
# TODO: Compute join and verify incidence
raise NotImplementedError("Implement exercise 2.1")

### ⭐⭐ Medium

**2.2** Create three lines forming a triangle: L1: x = 0, L2: y = 0, L3: x + y - 1 = 0. Find the three vertices using the meet operation. Verify each vertex lies on two of the three lines.

In [ ]:
# Your turn: ⭐⭐ Exercise 2.2
L1 = alg.multivector({"e1": 1.0, "e2": 0.0, "e0": 0.0})   # x = 0
L2 = alg.multivector({"e1": 0.0, "e2": 1.0, "e0": 0.0})   # y = 0
L3 = alg.multivector({"e1": 1.0, "e2": 1.0, "e0": -1.0}) # x + y - 1 = 0
# TODO: Find vertices using meet
raise NotImplementedError("Implement exercise 2.2")

### ⭐⭐⭐ Challenge

**2.3** Write a function `circle_through_three_points(p1, p2, p3)` that finds the center and radius of a circle passing through three non-collinear points. Use the meet and join operations. Hint: the perpendicular bisectors of the segments meet at the center.

In [ ]:
# Your turn: ⭐⭐⭐ Exercise 2.3
def circle_through_three_points(p1, p2, p3):
    """Find circle through three points. Return (center, radius)."""
    # TODO: Use perpendicular bisectors and meet
    raise NotImplementedError("Implement circle function")

# Test: equilateral triangle
c1 = alg.vector([0.0, 0.0, 1.0])
c2 = alg.vector([1.0, 0.0, 1.0])
c3 = alg.vector([0.5, np.sqrt(3)/2, 1.0])
center, radius = circle_through_three_points(c1, c2, c3)
print("Center:", center.values if center else "None")
print("Radius:", radius)

## Attribution

This notebook draws on:

- **Projective Geometric Algebra** — Charles G. Gunn
  https://arxiv.org/abs/1901.05873
- **SIGGRAPH 2019 Course Notes** — Charles G. Gunn
  https://arxiv.org/abs/1002.04509
- **PGABLE Tutorial** — Leger and Mann
  https://cs.uwaterloo.ca/~smann/PGABLE/PGAtutorial.pdf
- **Geometric Algebra for Computer Science** — Dorst, Lewiner, et al.
  https://geometricalgebra.org/